# Chapter 11 – Worked examples

This notebook runs the examples of [Chapter 11](https://katra.ballardini.com.ar/power-grid-topology-optimization-course/devdev/part-2-circuits/ch11-transformers-psts-hvdc/) live. The code lives in the modules next to it
(`dc_nodal.py`, `pst_flows.py`, `pst_phasors.py`, `tap_tables.py`, `trafo_equivalents.py`, `hvdc_ring.py`);
the cells only call them, so a change in a module is a change here too.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
# JupyterLite only: install the libraries this notebook and its chapter modules use.
%pip install -q matplotlib numpy pandas scipy tzdata

In [ ]:
import cmath
import math

import hvdc_ring
import matplotlib.pyplot as plt
import numpy as np
import pst_flows
import pst_phasors
import tap_tables
import trafo_equivalents

## The PST in a two-path network

$F_a = \dfrac{x_b}{x_a + x_b} P + \dfrac{\alpha}{x_a + x_b}$: a base share plus a loop flow. With $x_a = 0.1$,
$x_b = 0.3$ and $P = 1$ pu, `two_path_flows` gives $F_a = 0.75$ at $\alpha = 0$ and `slope_per_degree` the
0.0436 pu per degree (4.36 MW per degree on 100 MVA) that explains ToOp's `base_mva * (np.pi / 180)` scaling.

In [ ]:
f_a, f_b = pst_flows.two_path_flows(0.0)
print(f'alpha = 0: F_a = {f_a:.3f}, F_b = {f_b:.3f} pu')
print(f'slope: {pst_flows.slope_per_degree():.4f} pu/deg = {100 * pst_flows.slope_per_degree():.2f} MW/deg')
alpha = np.linspace(-10, 10, 201)
f_a, f_b = pst_flows.two_path_flows(alpha)
plt.plot(alpha, f_a, label='F_a (with the PST)')
plt.plot(alpha, f_b, label='F_b')
plt.axhline(0, color='k', linewidth=0.5)
plt.xlabel('alpha (deg)')
plt.ylabel('flow (pu)')
plt.legend()
plt.grid(True)
plt.show()
print(f'line b carries nothing at alpha = {pst_flows.alpha_for_zero_flow_b():.2f} deg')

The same PST as a pair of injections $\mp\alpha/x_a$ plus $\alpha/x_a$ added back to its own branch, solved
with the Chapter 8 `nodal_solve` (here in `dc_nodal.py`): identical flows.

In [ ]:
for a in (-5.0, 0.0, 5.0):
    print(a, np.round(pst_flows.two_path_flows(a), 4), np.round(pst_flows.two_path_by_injections(a), 4))

## Phasors of a quadrature booster and a symmetric PST

For the same shift the quadrature design lengthens the voltage by $1/\cos\alpha$; the symmetric design keeps it.

In [ ]:
u = 1 + 0j
for k in (0.05, 0.1, 0.2):
    w_q = pst_phasors.quadrature_booster(u, k)
    shift = math.degrees(cmath.phase(w_q))
    w_s = pst_phasors.symmetric_pst(u, shift)
    print(f'k = {k}: shift {shift:.2f} deg, |W_q| = {abs(w_q):.4f}, |W_s| = {abs(w_s):.4f}')

## Tap tables and the linearity test

`tap_tables.constant_x_table` has the same reactance at every tap and the atan2 shift; `growing_x_table` lets
the reactance grow 1 % per tap. `is_linear_pst` is the `np.allclose` test of ToOp's importer.

In [ ]:
for table in (tap_tables.constant_x_table(), tap_tables.growing_x_table()):
    print('linear in DC:', tap_tables.is_linear_pst(table))
tap_tables.constant_x_table().set_index('tap').loc[[-16, -8, -1, 0, 1, 8, 16]]

## Transformer equivalents

Referred impedance, ToOp's $x/\rho$ and the star of a three-winding transformer (Exercises 1 to 3).

In [ ]:
print(f'10 ohm at 110 kV from the 380 kV side: {trafo_equivalents.referred_impedance(10.0, 380.0, 110.0):.1f} ohm')
share_1, share_105 = trafo_equivalents.flow_share_with_ratio(1.0), trafo_equivalents.flow_share_with_ratio(1.05)
change = 100 * trafo_equivalents.susceptance_change(1.05)
print(f'rho = 1.05: susceptance {change:+.0f} %, flow share {share_1:.4f} -> {share_105:.4f}')
print('star equivalent:', trafo_equivalents.star_equivalent(0.10, 0.25, 0.20))

## An HVDC link as a pair of injections

Every AC flow of the three-bus ring is affine in $P_{dc}$ (`hvdc_ring.ac_flows`); `best_setpoint` finds the
transfer that minimizes the largest AC flow.

In [ ]:
p = np.linspace(-0.5, 1.5, 101)
flows = np.array([hvdc_ring.ac_flows(pk) for pk in p])
for k, name in enumerate(hvdc_ring.NAMES):
    plt.plot(p, flows[:, k], label=name)
p_best, f_max = hvdc_ring.best_setpoint()
plt.axvline(p_best, color='k', linestyle='--', label=f'best P_dc = {p_best:.3f}')
plt.xlabel('P_dc (pu)')
plt.ylabel('AC flow (pu)')
plt.legend()
plt.grid(True)
plt.show()
print(f'largest AC flow at the best setpoint: {f_max:.3f} pu')